In [ ]:
import json
import torch
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

DATASET_PATH = "/content/dataset.json"
SAIDA_PATH = "/content/drive/MyDrive/Prompt2 (1)/RespostasLlamaPrompt2.json"
HUGGINGFACE_TOKEN = "toke aqui"  # 🔑 seu token aqui

# ======================
# 🔹 CARREGAR TOKENIZER
# ======================
print("Carregando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    token=HUGGINGFACE_TOKEN
)


Carregando tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ======================
# 🔹 CARREGAR MODELO
# ======================
print("Carregando modelo llama...")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    token=HUGGINGFACE_TOKEN,  # ✅ CORREÇÃO: token também é necessário aqui
    device_map="auto",
    dtype=torch.bfloat16
)
print("Modelo carregado!")

Carregando modelo llama...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Modelo carregado!


In [4]:
# ======================
# 🔹 CARREGAR DATASET
# ======================
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

In [10]:
import json
import torch
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# ======================
# 🔹 MONTAR PROMPT
# ======================
def montar_prompt(questao):
    system = """Você é um especialista em Direito brasileiro e concursos públicos."""

    user = """Responda a questão de múltipla escolha.

    INSTRUÇÕES IMPORTANTES:
    - Leia o enunciado com atenção
    - Identifique o tema jurídico (Constitucional, Administrativo, Penal, etc.)
    - Baseie-se na legislação brasileira (especialmente Constituição e leis)
    - Analise TODAS as alternativas comparando com a regra correta
    - Elimine alternativas parcialmente corretas ou com erros sutis
    - Cuidado com prazos, exceções e termos técnicos (pegadinhas)
    - NÃO escolha a alternativa apenas por parecer correta — verifique se está totalmente correta

    REGRAS DE RESPOSTA:
    - NÃO explique
    - NÃO justifique
    - Responda apenas com uma letra (A, B, C, D ou E)
    """

    user += f"\nQUESTÃO:\n{questao['enunciado']}\n"

    if "itens" in questao and questao["itens"]:
        user += "\nITENS:\n"
        for item in questao["itens"]:
            user += f"{item['item']}) {item['texto']}\n"

    user += "\nALTERNATIVAS:\n"
    for letra, texto in questao["alternativas"].items():
        user += f"{letra}) {texto}\n"

    user += "\nResposta (apenas uma letra):"

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


# ======================
# 🔹 EXTRAIR E ANALISAR RESPOSTA
# ======================
def extrair_letra_e_analisar(resposta_modelo_completa):
    """
    Analisa a resposta bruta do modelo para classificar a resposta (A-E, MÚLTIPLAS, ERRO)
    e retorna uma versão truncada da resposta original para revisão.
    """
    # Capture a parte relevante da resposta do modelo para revisão
    resposta_para_registro = resposta_modelo_completa.strip()
    if len(resposta_para_registro) > 100: # Truncar se for muito longa
        resposta_para_registro = resposta_para_registro[:97] + "..."

    # Analisar o texto para encontrar letras de resposta válidas
    texto_analise = resposta_para_registro.upper()

    found_letters = set()
    for char in texto_analise:
        if char in ['A', 'B', 'C', 'D', 'E']:
            found_letters.add(char)

    if len(found_letters) == 1:
        classification = list(found_letters)[0]
    elif len(found_letters) > 1:
        # Adiciona as letras encontradas para maior detalhe na classificação
        classification = "MÚLTIPLAS (" + ", ".join(sorted(list(found_letters))) + ")"
    else:
        classification = "ERRO"

    return classification, resposta_para_registro

In [6]:
# ======================
# 🔹 INFERÊNCIA
# ======================
print("Iniciando inferência...")
resultados = []

for i, questao in enumerate(dataset):
    try:
        prompt = montar_prompt(questao)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                # ✅ CORREÇÃO: temperature removida — inválida com do_sample=False
                # e causa erro em versões recentes do Transformers
            )

        resposta_bruta = tokenizer.decode(output[0], skip_special_tokens=True)

        # ✅ CORREÇÃO: separar a parte do assistente da resposta bruta
        resposta_bruta_assistant_part = ""
        if "<|start_header_id|>assistant" in resposta_bruta:
            resposta_bruta_assistant_part = resposta_bruta.split("<|start_header_id|>assistant")[-1]
        elif "assistant" in resposta_bruta.lower(): # Fallback for older models or different templates
            resposta_bruta_assistant_part = resposta_bruta.split("assistant")[-1]
        else:
            resposta_bruta_assistant_part = resposta_bruta # Caso não encontre o separador

        # Use a nova função para classificar a resposta e obter a resposta bruta truncada
        classificacao_resposta, texto_resposta_modelo = extrair_letra_e_analisar(resposta_bruta_assistant_part)

        resultados.append({
            "id": questao["id"],
            "resposta_classificada": classificacao_resposta, # 'A', 'B', 'C', 'D', 'E', 'MÚLTIPLAS', 'ERRO'
            "resposta_modelo_bruta": texto_resposta_modelo # Texto real do modelo (truncado)
        })

        print_output = f"[{i+1}/{len(dataset)}] {questao['id']} -> {classificacao_resposta}"
        if classificacao_resposta == "MÚLTIPLAS":
            print_output += " (Múltiplas letras detectadas)"
        elif classificacao_resposta == "ERRO":
            print_output += " (Nenhuma letra válida detectada)"

        print_output += f" | Modelo respondeu: '{texto_resposta_modelo}'"
        print(print_output)

    except Exception as e:
        # ✅ CORREÇÃO: erros por questão não quebram o loop inteiro
        print(f"[{i+1}/{len(dataset)}] ERRO em {questao['id']}: {e}")
        resultados.append({
            "id": questao["id"],
            "resposta_classificada": "ERRO",
            "resposta_modelo_bruta": f"ERRO INTERNO: {e}"
        })

# ======================
# 🔹 SALVAR
# ======================
with open(SAIDA_PATH, "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print(f"\nRespostas salvas em: {SAIDA_PATH}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Iniciando inferência...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[1/266] pm2021_c_21 -> A | Modelo respondeu: 'A)'


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

[263/266] pc2025_04_lipc_63 -> B | Modelo respondeu: 'B)'


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[264/266] pc2025_04_lipc_64 -> A | Modelo respondeu: 'A'


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[265/266] pc2025_04_lipc_65 -> E | Modelo respondeu: 'E'
[266/266] pc2025_04_lipc_66 -> E | Modelo respondeu: 'E'

Respostas salvas em: /content/drive/MyDrive/Prompt2 (1)/RespostasLlamaPrompt2.json


In [7]:
# ======================
# 🔹 ANÁLISE DE CONFORMIDADE COM O PROMPT
# ======================
nao_respeitaram_prompt = 0
for res in resultados:
    if res['resposta_classificada'] in ['MÚLTIPLAS', 'ERRO']:
        nao_respeitaram_prompt += 1

print(f"\nTotal de questões analisadas: {len(resultados)}")
print(f"Questões que NÃO respeitaram o prompt (Múltiplas/Erro): {nao_respeitaram_prompt}")


Total de questões analisadas: 266
Questões que NÃO respeitaram o prompt (Múltiplas/Erro): 117


In [8]:
import os
import json
import pandas as pd

# 📁 Caminho dos gabaritos
CAMINHO_GABARITOS = "/content/drive/MyDrive/gabaritos"

# 📄 Arquivo JSON com respostas do modelo
ARQUIVO_JSON = "/content/drive/MyDrive/Prompt2 (1)/RespostasLlamaPrompt2.json"


def carregar_gabaritos(caminho):
    """
    Lê todos os CSVs e junta em um dicionário: {id: resposta}
    """
    gabarito = {}

    for arquivo in os.listdir(caminho):
        if arquivo.endswith(".csv"):
            caminho_arquivo = os.path.join(caminho, arquivo)
            df = pd.read_csv(caminho_arquivo)

            for _, row in df.iterrows():
                gabarito[row["id"]] = str(row["resposta"]).strip().upper()

    return gabarito


def carregar_respostas_json(caminho_json):
    """
    Lê o JSON com respostas do modelo
    """
    with open(caminho_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    respostas = {}
    for item in dados:
        # ✅ CORREÇÃO: Usar a resposta classificada para comparação
        if "resposta_classificada" in item:
            respostas[item["id"]] = item["resposta_classificada"].strip().upper()
        else:
            # Fallback for old format if necessary, though it should be updated
            respostas[item["id"]] = item["resposta"].strip().upper()

    return respostas


def comparar_respostas(gabarito, respostas_modelo):
    """
    Compara respostas e gera estatísticas
    """
    acertos = 0
    erros = 0
    nao_encontradas = 0

    detalhes = []

    for id_pergunta, resposta_modelo in respostas_modelo.items():
        if id_pergunta not in gabarito:
            nao_encontradas += 1
            detalhes.append((id_pergunta, resposta_modelo, "N/A", "SEM GABARITO"))
            continue

        resposta_correta = gabarito[id_pergunta]

        if resposta_modelo == resposta_correta:
            acertos += 1
            status = "ACERTO"
        else:
            erros += 1
            status = "ERRO"

        detalhes.append((id_pergunta, resposta_modelo, resposta_correta, status))

    total = acertos + erros

    acuracia = acertos / total if total > 0 else 0

    return {
        "acertos": acertos,
        "erros": erros,
        "nao_encontradas": nao_encontradas,
        "total_avaliado": total,
        "acuracia": acuracia,
        "detalhes": detalhes
    }


def salvar_resultado(resultado, arquivo_saida="/content/drive/MyDrive/Prompt2 (1)/ResultadosLlamaPrompt2.csv"):
    """
    Salva os detalhes da comparação em CSV
    """
    df = pd.DataFrame(
        resultado["detalhes"],
        columns=["id", "resposta_modelo", "resposta_correta", "status"]
    )

    df.to_csv(arquivo_saida, index=False)
    print(f" Resultado salvo em: {arquivo_saida}")

In [9]:
print(" Carregando gabaritos...")
gabarito = carregar_gabaritos(CAMINHO_GABARITOS)

print(" Carregando respostas do JSON...")
respostas_modelo = carregar_respostas_json(ARQUIVO_JSON)

print(" Comparando...")
resultado = comparar_respostas(gabarito, respostas_modelo)

print("\nRESULTADO FINAL")
print(f" Acertos: {resultado['acertos']}")
print(f" Erros: {resultado['erros']}")
print(f" Sem gabarito: {resultado['nao_encontradas']}")
print(f" Acurácia: {resultado['acuracia']:.2%}")

salvar_resultado(resultado)


 Carregando gabaritos...
 Carregando respostas do JSON...
 Comparando...

RESULTADO FINAL
 Acertos: 17
 Erros: 249
 Sem gabarito: 0
 Acurácia: 6.39%
 Resultado salvo em: /content/drive/MyDrive/Prompt2 (1)/ResultadosLlamaPrompt2.csv
